In [1]:
from langchain_groq import ChatGroq

In [3]:
llm = ChatGroq(
    temperature=0, 
    groq_api_key='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', 
    model_name="openai/gpt-oss-120b"
)
response = llm.invoke("The first person to land on moon was ...")
print(response.content)

The first person to set foot on the Moon was **Neil Armstrong**, who did so on July 20 1969 during NASA’s Apollo 11 mission. He famously described the moment as “one small step for [a] man, one giant leap for mankind.”


In [4]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://careers.nike.com/principal-operations-spain/job/R-91046")
page_data = loader.load().pop().page_content
print(page_data)

C:\Users\Subho\AppData\Local\Temp\ipykernel_11032\1709153204.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.






















Principal, Operations Spain














































Skip to main content
Open Virtual Assistant










Home


Career Areas


Total Rewards


Life@Nike


Purpose










Language





Select a Language

  Deutsch  
  English  
  Español (España)  
  Español (América Latina)  
  Français  
  Italiano  
  Nederlands  
  Polski  
  Tiếng Việt  
  Türkçe  
  简体中文  
  繁體中文  
  עִברִית  
  한국어  
  日本語  








Careers


















Close Menu







Careers






Chat






                                Home
                            



                                Career Areas
                            



                                Total Rewards
                            



                                Life@Nike
                            



                                Purpose
                            










Jordan Careers







Converse Careers










Language











Menu



Return to Previous Men

In [5]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
        """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the 
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):    
        """
)

chain_extract = prompt_extract | llm 
res = chain_extract.invoke(input={'page_data':page_data})
type(res.content)

str

In [6]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

{'role': 'Principal, Operations Spain',
 'experience': ['Significant experience in Integrated Business Planning, Supply & Operations Planning or Business Execution',
  'Proven ability to drive operational performance improvements across functions',
  'Experience driving adoption of enterprise tools, automation and standardized processes',
  'Strong business acumen and data‑driven decision‑making skills'],
 'skills': ['Integrated Business Planning (IBP)',
  'Supply & Operations Planning (S&OP)',
  'Business Execution',
  'Data‑driven decision making',
  'Stakeholder management and communication',
  'Analytical and structured problem solving',
  'Enterprise tool adoption and automation',
  'AI‑enabled ways of working',
  'Process simplification and efficiency improvement',
  'Cross‑functional partnership and influencing without authority'],
 'description': 'As a member of the Country Leadership Team, the Principal, Operations Spain will translate strategy into execution by leading integr

In [7]:
type(json_res)

dict

In [8]:
import pandas as pd

df = pd.read_csv("my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [9]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

In [ ]:
job = json_res
links = collection.query(query_texts=job['skills'], n_results=2).get('metadatas', [])
links

NameError: name 'job' is not defined

In [ ]:
job

{'role': 'Software Engineer II, AI/ML Platforms',
 'experience': '2+ years professional software development experience',
 'skills': ['Strong Computer Science fundamentals',
  'Programming experience with at least one modern language such as Java, Python, Golang',
  'Experience with Infrastructure as Code / DevOps practices and technologies like Terraform',
  'Experience with Cloud services (e.g. AWS Cloud EC2, S3, DynamoDB, Azure Cloud, etc.)',
  'Experience developing and using microservices'],
 'description': 'We are looking for a Software Engineer II to design and implement core components of our data science services and platform and scale it to serve Nike globally. The focus of this engineer will be at the intersection of computer systems and machine learning. You will also work collaboratively with the data science and analytics teams to ship scalable, performant and reliable solutions.'}

In [ ]:
job = json_res
job['skills']

['Strong Computer Science fundamentals',
 'Programming experience with at least one modern language such as Java, Python, Golang',
 'Experience with Infrastructure as Code / DevOps practices and technologies like Terraform',
 'Experience with Cloud services (e.g. AWS Cloud EC2, S3, DynamoDB, Azure Cloud, etc.)',
 'Experience developing and using microservices']

In [ ]:
prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}
        
        ### INSTRUCTION:
        You are Mohan, a business development executive at AtliQ. AtliQ is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools. 
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability, 
        process optimization, cost reduction, and heightened overall efficiency. 
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of AtliQ 
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase Atliq's portfolio: {link_list}
        Remember you are Mohan, BDE at AtliQ. 
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):
        
        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Expert AI/ML Solutions for Nike's Data Science Services and Platform

Dear Hiring Manager,

I came across the job description for a Software Engineer II, AI/ML Platforms at Nike, and I'm excited to introduce AtliQ, a leading AI & Software Consulting company that can help you design and implement core components of your data science services and platform.

With our expertise in AI, ML, and software development, we can help you scale your platform to serve Nike globally. Our team of experts has a strong foundation in computer science and experience with modern programming languages such as Java, Python, and Golang. We're well-versed in Infrastructure as Code / DevOps practices and technologies like Terraform, and have hands-on experience with Cloud services like AWS and Azure.

Our portfolio showcases our capabilities in developing and deploying scalable, performant, and reliable solutions. I'd like to highlight a few relevant examples:

* Our Machine Learning with Python portfo